# GPU RF + Autoencoder Latent-Index Shared-Book ML Trading

Train collapsed-label CUDA random forest classifiers and one autoencoder per feature family on all available pre-2020 Quant Warehouse data, then trade the 2020+ out-of-sample predictions with one shared multi-asset book.

This notebook tests only `classifier + autoencoder latent familiarity`. The classifier-only baseline lives in `gpu_rf_shared_book_ml_trading.ipynb` and is used as the comparison reference in the analysis.

The autoencoder is not scored by reconstruction error. Each feature family gets its own in-memory latent nearest-neighbor index built from the pre-2020 training rows. Familiarity is computed from the distance between each candidate row's latent vector and that feature family's training latent vectors.

Both entries and exits use classifier probabilities multiplied by AE latent familiarity. That mirrors the original optimal-trader idea: the model should prefer trades whose entry and exit states are close to states it has learned in latent space.

The notebook compares long-only, short-only, and shared long+short variants for `top_k = [5, 10, 20, 40]`. Each feature family is traded by itself as a strategy source, and the ensemble mean is also traded. Execution uses native Zipline multi-asset shared-book orders through `order_target_percent`. There is no arbitrary trade cap; every eligible score row is passed into signal generation, subject only to the intentional portfolio `top_k` capacity and score thresholds.


In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder

QUANT_WAREHOUSE_ROOT = Path('/home/jlee153232/PycharmProjects/quant-warehouse')
if str(QUANT_WAREHOUSE_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANT_WAREHOUSE_ROOT))

import cupy as cp
import cudf
from cuml.ensemble import RandomForestClassifier as CuRandomForestClassifier

from quant_orchestrator.platforms.backtesting_frameworks.shared_book import build_shared_book_weights
from quant_orchestrator.platforms.backtesting_frameworks.zipline.shared_book import (
    ZiplineSharedBookSummaryJob,
    run_zipline_shared_book_summary_jobs,
)

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_collapsed_bullish_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
warnings.filterwarnings('ignore', category=FutureWarning)

print('cupy', cp.__version__, 'cuda_devices', cp.cuda.runtime.getDeviceCount())
print('cudf', cudf.__version__)

cupy 14.1.1 cuda_devices 1
cudf 26.06.00


In [2]:
RANDOM_SEED = 20260702
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '1900-01-01'
END_DATE = None
TRAIN_END = pd.Timestamp('2019-12-31')
OOS_START = pd.Timestamp('2020-01-01')

TOP_K_VALUES = [5, 10, 20, 40]
ENTRY_THRESHOLD = 0.50
EXIT_THRESHOLD = 0.50
MIN_FEATURE_COVERAGE = 0.50
MAX_FEATURES_PER_FAMILY = 50
MIN_TRAIN_ROWS_PER_FAMILY = 250
MIN_CLASSES_PER_FAMILY = 2
CAPITAL_BASE = 1_000_000.0

ZIPLINE_COMMISSION_PER_SHARE = 0.005
ZIPLINE_SLIPPAGE_BPS = 5.0
ZIPLINE_MAX_WORKERS = 4

AE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AE_EPOCHS = 20
AE_BATCH_SIZE = 4096
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-4
AE_DENOISE_STD = 0.02
AE_FAMILIARITY_QUANTILE = 99.9
AE_NN_METRIC = 'euclidean'

RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 16,
    'max_features': 'sqrt',
    'n_bins': 128,
    'random_state': RANDOM_SEED,
    'n_streams': 8,
}

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    max_features_per_family=MAX_FEATURES_PER_FAMILY,
)
TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=START_DATE,
    end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('torch', torch.__version__, 'ae_device', AE_DEVICE)
FEATURE_CONFIG, TARGET_CONFIG

torch 2.12.1+cu130 ae_device cuda


(FamilyEvaluationConfig(provider='fmp', market_cap_min=1000000000000, country='US', exchanges=('NASDAQ', 'NYSE', 'AMEX'), screen_limit=5000, start_date='1900-01-01', end_date=None, filing_lag_days=45, horizons=(20, 60, 120), min_observations=120, max_features_per_family=50),
 BinaryTargetConfig(provider='fmp', start_date='1900-01-01', end_date=None, event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'), oracle_trade_k_by_frequency={'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}, oracle_trade_min_profit_pct=0.01, oracle_trade_long_entry_price_col='high', oracle_trade_long_exit_price_col='low', oracle_trade_short_entry_price_col='low', oracle_trade_short_exit_price_col='high', event_alignment_tolerance_days=7, collapsed_bullish_event_types=('congress_buy', 'insider_buy', 'analyst_upgrade', 'price_target_raise', 'earnings_beat')))

## Build Data

Training uses every available row before 2020. Out-of-sample classification metrics and trading results both use 2020+ rows.

In [3]:
started = perf_counter()
WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(backend=WAREHOUSE.backend, catalog=WAREHOUSE.catalog)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)
print({'universe_source': universe_source, 'eligible_symbols': len(symbols)})
display(universe_eligibility.loc[universe_eligibility['eligible']].head(30))

{'universe_source': 'openbb:fmp', 'eligible_symbols': 14}


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols, FEATURE_CONFIG, warehouse=WAREHOUSE
)
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel, raw_feature_metadata, max_features=MAX_FEATURES_PER_FAMILY
)
feature_panel = raw_feature_panel[['symbol', 'date', *selected_features]].copy()
feature_panel['symbol'] = feature_panel['symbol'].astype(str).str.upper()
feature_panel['date'] = pd.to_datetime(feature_panel['date'], errors='coerce').dt.normalize()
print({
    'raw_feature_panel_rows': len(raw_feature_panel),
    'selected_feature_columns': len(selected_features),
    'selected_feature_families': selected_feature_metadata['family'].nunique(),
    **feature_timings,
})
display(selected_feature_metadata.groupby(['source', 'family'], as_index=False).agg(feature_count=('feature', 'nunique')).sort_values(['source', 'family']))

{'raw_feature_panel_rows': 100125, 'selected_feature_columns': 365, 'selected_feature_families': 15, 'raw_panel_build_seconds': 1.4599785022437572}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols, TARGET_CONFIG, event_store=EVENT_STORE, include_historical=True
)
event_symbols = tuple(event_diagnostics.loc[event_diagnostics['combined_rows'].gt(0), 'symbol'].sort_values())
feature_panel = feature_panel.loc[feature_panel['symbol'].isin(event_symbols)].copy()

collapsed_event_panel, collapsed_event_metadata = build_collapsed_bullish_event_target_panel(
    feature_panel[['symbol', 'date']], events, TARGET_CONFIG
)
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols, TARGET_CONFIG, warehouse=WAREHOUSE
)
print({
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'event_load_seconds': round(event_load_seconds, 3),
    'collapsed_event_rows': len(collapsed_event_panel),
    'oracle_rows': len(oracle_panel),
    'oracle_columns': len(oracle_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
})
display(event_diagnostics.sort_values('combined_rows', ascending=False).head(20))

{'event_symbols': 13, 'event_rows': 5836, 'event_load_seconds': 0.895, 'collapsed_event_rows': 100114, 'oracle_rows': 100114, 'oracle_columns': 36, 'oracle_seconds': 3.934}


,symbol,cached_rows,historical_rows,combined_rows,event_families
9,MSFT,65,971,1036,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,62,885,947,"(analyst_rating, congress, earnings, insider, ..."
11,NVDA,47,812,859,"(analyst_rating, congress, earnings, insider, ..."
1,AMZN,66,742,808,"(analyst_rating, congress, earnings, insider, ..."
6,GOOGL,60,590,650,"(analyst_rating, congress, earnings, insider, ..."
13,TSLA,60,459,519,"(analyst_rating, congress, earnings, insider, ..."
8,META,55,437,492,"(analyst_rating, congress, earnings, insider, ..."
10,MU,0,121,121,"(earnings,)"
7,LLY,0,104,104,"(earnings,)"
3,BRK-A,0,102,102,"(earnings,)"


In [6]:
def collapsed_label_rows(collapsed_event_panel: pd.DataFrame, oracle_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    event_target = 'target_event_collapsed__bullish'
    event_activity = f'_target_activity__{event_target}'
    frames = []
    diagnostics = []

    event_frame = collapsed_event_panel.copy()
    if event_target in event_frame.columns:
        activity = pd.to_numeric(event_frame.get(event_activity, 0), errors='coerce').fillna(0).astype('int8')
        value = pd.to_numeric(event_frame[event_target], errors='coerce').fillna(0).astype('int8')
        bullish = event_frame.loc[value.eq(1), ['symbol', 'date']].copy()
        bullish['collapsed_label'] = 'event_bullish'
        bullish['label_source'] = 'event_collapsed'
        frames.append(bullish)
        diagnostics.append({'source': 'event_collapsed', 'candidate_rows': int(activity.gt(0).sum()), 'used_rows': len(bullish), 'dropped_rows': int(activity.gt(0).sum()) - len(bullish), 'note': 'mirror/non-bullish event rows excluded'})

    long_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_long'))
    short_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_short'))
    if long_cols and short_cols:
        oracle = oracle_panel[['symbol', 'date', *long_cols, *short_cols]].copy()
        long_any = oracle[long_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        short_any = oracle[short_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        ambiguous = long_any & short_any
        long_rows = oracle.loc[long_any & ~short_any, ['symbol', 'date']].copy()
        long_rows['collapsed_label'] = 'oracle_long'
        long_rows['label_source'] = 'oracle_trade'
        short_rows = oracle.loc[short_any & ~long_any, ['symbol', 'date']].copy()
        short_rows['collapsed_label'] = 'oracle_short'
        short_rows['label_source'] = 'oracle_trade'
        frames.extend([long_rows, short_rows])
        diagnostics.append({'source': 'oracle_trade', 'candidate_rows': int((long_any | short_any).sum()), 'used_rows': len(long_rows) + len(short_rows), 'dropped_rows': int(ambiguous.sum()), 'note': 'ambiguous long+short rows dropped after k collapse'})

    labels = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['symbol', 'date', 'collapsed_label', 'label_source'])
    labels['symbol'] = labels['symbol'].astype(str).str.upper()
    labels['date'] = pd.to_datetime(labels['date'], errors='coerce').dt.normalize()
    labels = labels.dropna(subset=['symbol', 'date', 'collapsed_label']).drop_duplicates()
    return labels.sort_values(['date', 'symbol', 'collapsed_label']).reset_index(drop=True), pd.DataFrame(diagnostics)

label_rows, label_diagnostics = collapsed_label_rows(collapsed_event_panel, oracle_panel)
print({'collapsed_label_rows': len(label_rows), 'classes': sorted(label_rows['collapsed_label'].unique())})
display(label_diagnostics)
display(label_rows.groupby(['label_source', 'collapsed_label'], as_index=False).agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max')))

{'collapsed_label_rows': 11581, 'classes': ['event_bullish', 'oracle_long', 'oracle_short']}


,source,candidate_rows,used_rows,dropped_rows,note
0,event_collapsed,4131,2353,1778,mirror/non-bullish event rows excluded
1,oracle_trade,9228,9228,0,ambiguous long+short rows dropped after k coll...


,label_source,collapsed_label,rows,symbols,min_date,max_date
0,event_collapsed,event_bullish,2353,13,1993-03-26,2026-06-22
1,oracle_trade,oracle_long,4677,13,1970-07-17,2026-06-12
2,oracle_trade,oracle_short,4551,13,1970-07-09,2026-06-22


## Train GPU RF Family Models Pre-2020

Each feature family gets one CUDA random forest. Out-of-sample metrics are measured on 2020+ labeled rows.

In [7]:
def prepare_family_dataset(feature_panel, feature_metadata, labels, *, source, family, min_feature_coverage):
    family_meta = feature_metadata.loc[feature_metadata['source'].astype(str).eq(source) & feature_metadata['family'].astype(str).eq(family)]
    features = [f for f in family_meta['feature'].drop_duplicates().tolist() if f in feature_panel.columns]
    numeric_features = [f for f in features if pd.to_numeric(feature_panel[f], errors='coerce').notna().any()]
    if not numeric_features:
        return pd.DataFrame(), []
    merged = labels.merge(feature_panel[['symbol', 'date', *numeric_features]], on=['symbol', 'date'], how='inner')
    if merged.empty:
        return merged, numeric_features
    numeric = merged[numeric_features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    merged = merged.loc[coverage.ge(min_feature_coverage)].copy()
    if merged.empty:
        return merged, numeric_features
    numeric = numeric.loc[merged.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    merged[numeric_features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return merged.reset_index(drop=True), numeric_features


def fit_gpu_random_forest(train, features):
    encoder = LabelEncoder()
    y = encoder.fit_transform(train['collapsed_label'].astype(str))
    model = CuRandomForestClassifier(**RF_PARAMS)
    model.fit(cudf.from_pandas(train[features].astype('float32')), cudf.Series(y.astype('int32')))
    return model, encoder


def predict_proba_frame(model, encoder, frame, features):
    proba = model.predict_proba(cudf.from_pandas(frame[features].astype('float32')))
    proba_np = proba.to_numpy() if hasattr(proba, 'to_numpy') else cp.asnumpy(proba)
    out = pd.DataFrame(proba_np, columns=[f'prob__{label}' for label in encoder.classes_], index=frame.index)
    for label in ['event_bullish', 'oracle_long', 'oracle_short']:
        col = f'prob__{label}'
        if col not in out.columns:
            out[col] = 0.0
    return out


def score_classifier(model, encoder, frame, features):
    proba = predict_proba_frame(model, encoder, frame, features)
    y_true = frame['collapsed_label'].astype(str).to_numpy()
    y_pred = encoder.inverse_transform(np.asarray(proba[[f'prob__{label}' for label in encoder.classes_]].to_numpy().argmax(axis=1)).astype(int))
    return {
        'rows': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


class FamilyAutoEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, bottleneck_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def _standardize_fit(frame, features):
    raw = frame[features].apply(pd.to_numeric, errors='coerce').to_numpy(dtype='float64', copy=True)
    lower = np.nanpercentile(raw, 0.1, axis=0)
    upper = np.nanpercentile(raw, 99.9, axis=0)
    clipped = np.clip(raw, lower, upper)
    center = np.nanmedian(clipped, axis=0)
    q1 = np.nanpercentile(clipped, 25.0, axis=0)
    q3 = np.nanpercentile(clipped, 75.0, axis=0)
    scale = q3 - q1
    center = np.nan_to_num(center, nan=0.0, posinf=0.0, neginf=0.0)
    scale = np.where(np.isfinite(scale) & (scale > 1e-9), scale, 1.0)
    lower = np.nan_to_num(lower, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    upper = np.nan_to_num(upper, nan=np.inf, posinf=np.inf, neginf=-np.inf)
    filled = np.where(np.isfinite(clipped), clipped, center)
    x = (filled - center) / scale
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = np.clip(x, -50.0, 50.0).astype('float32')
    return x, center.astype('float32'), scale.astype('float32'), lower.astype('float32'), upper.astype('float32')


def _standardize_apply(frame, features, center, scale, lower, upper):
    raw = frame[features].apply(pd.to_numeric, errors='coerce').to_numpy(dtype='float64', copy=True)
    clipped = np.clip(raw, lower, upper)
    filled = np.where(np.isfinite(clipped), clipped, center)
    x = (filled - center) / scale
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(x, -50.0, 50.0).astype('float32')


def _autoencoder_recon_error(model, x):
    out = np.empty((len(x),), dtype='float64')
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x), AE_BATCH_SIZE):
            end = min(start + AE_BATCH_SIZE, len(x))
            batch = torch.tensor(x[start:end], dtype=torch.float32, device=AE_DEVICE)
            recon = model(batch).detach().cpu().numpy().astype('float32')
            diff = recon - x[start:end]
            out[start:end] = np.mean(diff * diff, axis=1)
    return out


def _autoencoder_latent(model, x):
    out = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x), AE_BATCH_SIZE):
            end = min(start + AE_BATCH_SIZE, len(x))
            batch = torch.tensor(x[start:end], dtype=torch.float32, device=AE_DEVICE)
            latent = model.encoder(batch).detach().cpu().numpy().astype('float32')
            out.append(latent)
    return np.vstack(out) if out else np.empty((0, 0), dtype='float32')


def fit_autoencoder(train, features):
    x_train, center, scale, lower, upper = _standardize_fit(train, features)
    if len(x_train) < 2:
        return None
    in_dim = x_train.shape[1]
    hidden_dim = max(8, min(128, in_dim * 2))
    bottleneck_dim = max(2, min(32, hidden_dim // 4, max(2, in_dim // 2)))
    model = FamilyAutoEncoder(in_dim, hidden_dim, bottleneck_dim).to(AE_DEVICE)
    loader = DataLoader(TensorDataset(torch.tensor(x_train, dtype=torch.float32)), batch_size=AE_BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    model.train()
    for _epoch in range(AE_EPOCHS):
        for (batch,) in loader:
            batch = batch.to(AE_DEVICE)
            noisy = batch + torch.randn_like(batch) * AE_DENOISE_STD if AE_DENOISE_STD > 0 else batch
            optimizer.zero_grad(set_to_none=True)
            recon = model(noisy)
            loss = loss_fn(recon, batch)
            if torch.isfinite(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
    model.eval()
    train_error = _autoencoder_recon_error(model, x_train)
    train_latent = _autoencoder_latent(model, x_train)
    nn_index = NearestNeighbors(n_neighbors=1, metric=AE_NN_METRIC)
    nn_index.fit(train_latent)
    calibration_neighbors = min(2, len(train_latent))
    calibration_index = NearestNeighbors(n_neighbors=calibration_neighbors, metric=AE_NN_METRIC)
    calibration_index.fit(train_latent)
    calibration_distance, _ = calibration_index.kneighbors(train_latent, return_distance=True)
    if calibration_neighbors > 1:
        train_latent_distance = calibration_distance[:, 1]
    else:
        train_latent_distance = calibration_distance[:, 0]
    latent_cutoff = float(np.nanpercentile(train_latent_distance, AE_FAMILIARITY_QUANTILE))
    latent_cutoff = max(latent_cutoff, 1e-12)
    if AE_DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return {
        'model': model,
        'features': list(features),
        'center': center,
        'scale': scale,
        'lower': lower,
        'upper': upper,
        'nn_index': nn_index,
        'latent_index_rows': int(len(train_latent)),
        'latent_distance_cutoff': latent_cutoff,
        'train_latent_distance_mean': float(np.nanmean(train_latent_distance)),
        'train_latent_distance_p95': float(np.nanpercentile(train_latent_distance, 95.0)),
        'train_error_mean': float(np.nanmean(train_error)),
        'train_error_p95': float(np.nanpercentile(train_error, 95.0)),
        'hidden_dim': hidden_dim,
        'bottleneck_dim': bottleneck_dim,
    }


def score_autoencoder_familiarity(ae_payload, frame, features):
    if ae_payload is None:
        zeros = np.zeros(len(frame), dtype='float64')
        ones = np.ones(len(frame), dtype='float64')
        return ones, zeros, zeros
    x = _standardize_apply(frame, features, ae_payload['center'], ae_payload['scale'], ae_payload['lower'], ae_payload['upper'])
    err = _autoencoder_recon_error(ae_payload['model'], x)
    latent = _autoencoder_latent(ae_payload['model'], x)
    latent_distance, _ = ae_payload['nn_index'].kneighbors(latent, n_neighbors=1, return_distance=True)
    latent_distance = latent_distance[:, 0].astype('float64')
    fam = 1.0 / (1.0 + (latent_distance / ae_payload['latent_distance_cutoff']))
    return np.clip(fam, 0.0, 1.0).astype('float64'), err.astype('float64'), latent_distance


model_rows = []
models = {}
train_started = perf_counter()
for source, family in selected_feature_metadata[['source', 'family']].drop_duplicates().sort_values(['source', 'family']).itertuples(index=False, name=None):
    family_frame, features = prepare_family_dataset(feature_panel, selected_feature_metadata, label_rows, source=str(source), family=str(family), min_feature_coverage=MIN_FEATURE_COVERAGE)
    if family_frame.empty:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_empty', 'features': len(features), 'rows': 0})
        continue
    train = family_frame.loc[pd.to_datetime(family_frame['date']).le(TRAIN_END)].copy()
    oos = family_frame.loc[pd.to_datetime(family_frame['date']).ge(OOS_START)].copy()
    if len(train) < MIN_TRAIN_ROWS_PER_FAMILY or train['collapsed_label'].nunique() < MIN_CLASSES_PER_FAMILY:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_sparse_train', 'features': len(features), 'rows': len(family_frame), 'train_rows': len(train), 'oos_rows': len(oos), 'train_classes': train['collapsed_label'].nunique()})
        continue
    fit_started = perf_counter()
    model, encoder = fit_gpu_random_forest(train, features)
    classifier_fit_seconds = perf_counter() - fit_started
    ae_started = perf_counter()
    autoencoder = fit_autoencoder(train, features)
    ae_fit_seconds = perf_counter() - ae_started
    models[(source, family)] = {'model': model, 'encoder': encoder, 'autoencoder': autoencoder, 'features': features}
    train_scores = score_classifier(model, encoder, train, features)
    oos_scores = score_classifier(model, encoder, oos, features) if not oos.empty and oos['collapsed_label'].nunique() > 1 else {'rows': len(oos), 'accuracy': np.nan, 'balanced_accuracy': np.nan, 'macro_f1': np.nan}
    model_rows.append({
        'source': source,
        'family': family,
        'status': 'ok',
        'features': len(features),
        'rows': len(family_frame),
        'train_rows': len(train),
        'oos_rows': len(oos),
        'classes': family_frame['collapsed_label'].nunique(),
        'classifier_fit_seconds': classifier_fit_seconds,
        'ae_fit_seconds': ae_fit_seconds,
        'ae_latent_index_rows': np.nan if autoencoder is None else autoencoder['latent_index_rows'],
        'ae_latent_distance_cutoff': np.nan if autoencoder is None else autoencoder['latent_distance_cutoff'],
        'ae_train_latent_distance_mean': np.nan if autoencoder is None else autoencoder['train_latent_distance_mean'],
        'ae_train_latent_distance_p95': np.nan if autoencoder is None else autoencoder['train_latent_distance_p95'],
        'ae_train_error_mean': np.nan if autoencoder is None else autoencoder['train_error_mean'],
        'ae_train_error_p95': np.nan if autoencoder is None else autoencoder['train_error_p95'],
        'ae_bottleneck_dim': np.nan if autoencoder is None else autoencoder['bottleneck_dim'],
        **{f'train_{k}': v for k, v in train_scores.items()},
        **{f'oos_{k}': v for k, v in oos_scores.items()},
    })

model_results = pd.DataFrame(model_rows).sort_values(['status', 'oos_macro_f1', 'oos_balanced_accuracy'], ascending=[True, False, False]).reset_index(drop=True)
print({'trained_models': len(models), 'elapsed_seconds': round(perf_counter() - train_started, 3), 'ae_device': AE_DEVICE, 'ae_nn_metric': AE_NN_METRIC})
display(model_results)


[22:37:32] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[22:37:34] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[22:37:36] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[22:37:38] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[22:37:40] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


...

[22:37:59] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[22:38:01] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


{'trained_models': 15, 'elapsed_seconds': 35.577, 'ae_device': 'cuda', 'ae_nn_metric': 'euclidean'}


[22:38:04] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,source,family,status,features,rows,train_rows,oos_rows,classes,classifier_fit_seconds,ae_fit_seconds,ae_latent_index_rows,ae_latent_distance_cutoff,ae_train_latent_distance_mean,ae_train_latent_distance_p95,ae_train_error_mean,ae_train_error_p95,ae_bottleneck_dim,train_accuracy,train_balanced_accuracy,train_macro_f1,oos_accuracy,oos_balanced_accuracy,oos_macro_f1
0,financetoolkit,ft_ratios_valuation,ok,24,10885,7126,3759,3,1.6907,0.2831,7126,0.0000,0.0001,0.0000,55.8927,510.2716,12,0.5265,0.4494,0.4688,0.4302,0.3953,0.3912
1,financetoolkit,ft_ratios_solvency,ok,15,10885,7126,3759,3,1.6750,0.2528,7126,0.0000,0.0000,0.0000,35.0185,161.4000,7,0.5236,0.4464,0.4660,0.2934,0.3448,0.2676
2,fmp,fmp_cash_mcap,ok,39,10430,6671,3759,3,1.7231,0.5911,6671,1.2550,0.0363,0.1549,30.7018,110.9803,19,0.9319,0.8437,0.8741,0.2969,0.3642,0.2555
3,financetoolkit,ft_ratios_efficiency,ok,5,10885,7126,3759,3,1.7276,0.2420,7126,0.0000,0.0000,0.0000,4.7516,13.7440,2,0.5240,0.4456,0.4654,0.2854,0.3410,0.2515
4,financetoolkit,ft_ratios_profitability,ok,14,10885,7126,3759,3,1.6652,0.5029,7126,0.0000,0.0000,0.0000,3.4742,13.1972,7,0.5261,0.4464,0.4659,0.2825,0.3402,0.2496
5,fmp,fmp_daily_ev_yield,ok,7,10885,7126,3759,3,1.7060,0.2446,7126,0.2927,0.0077,0.0287,6.3368,13.7378,3,0.9145,0.8062,0.8417,0.2876,0.3533,0.2486
6,fmp,fmp_daily_mcap_yield,ok,14,10826,7067,3759,3,1.8103,0.2791,7067,0.8087,0.0255,0.1016,3.0618,7.8206,7,0.9530,0.8749,0.9026,0.2961,0.3635,0.2480
7,fmp,fmp_balance_mcap,ok,50,10826,7067,3759,3,1.6813,0.2928,7067,0.9060,0.0422,0.1650,17.0206,65.6598,25,0.9458,0.8676,0.8971,0.2985,0.3607,0.2423
8,fmp,fmp_income_mcap,ok,31,10885,7126,3759,3,1.7780,0.3173,7126,1.1184,0.0362,0.1547,13.5925,65.4240,15,0.9499,0.8769,0.9026,0.2966,0.3588,0.2413
9,fmp,fmp_daily_ev_multiple,ok,7,10822,7063,3759,3,1.7609,0.2274,7063,0.2452,0.0054,0.0209,22.6749,103.2699,3,0.9216,0.8145,0.8504,0.2857,0.3524,0.2413


## Inference Scores

Each strategy source is classifier + autoencoder latent familiarity. The per-family latent index gates both entry scores and exit scores.

In [8]:
def prepare_prediction_frame(feature_panel, features, min_feature_coverage):
    base_cols = ['symbol', 'date']
    numeric = feature_panel[features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    out = feature_panel.loc[coverage.ge(min_feature_coverage), base_cols].copy()
    if out.empty:
        return out
    numeric = numeric.loc[out.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return out.reset_index(drop=True)


def strategy_source_name(source, family):
    return f'{source}.{family}'


pred_frames = []
for (source, family), payload in models.items():
    pred_input = prepare_prediction_frame(feature_panel, payload['features'], MIN_FEATURE_COVERAGE)
    if pred_input.empty:
        continue
    proba = predict_proba_frame(payload['model'], payload['encoder'], pred_input, payload['features'])
    ae_familiarity, ae_recon_error, ae_latent_distance = score_autoencoder_familiarity(payload['autoencoder'], pred_input, payload['features'])
    classifier_long = (proba['prob__event_bullish'] + proba['prob__oracle_long']).clip(0, 1).to_numpy(dtype='float64')
    classifier_short = proba['prob__oracle_short'].clip(0, 1).to_numpy(dtype='float64')
    scored = pred_input[['symbol', 'date']].copy()
    scored['source'] = str(source)
    scored['family'] = str(family)
    scored['strategy_source'] = strategy_source_name(source, family)
    scored['classifier_long_score'] = classifier_long
    scored['classifier_short_score'] = classifier_short
    scored['ae_familiarity'] = ae_familiarity
    scored['ae_recon_error'] = ae_recon_error
    scored['ae_latent_distance'] = ae_latent_distance
    scored['long_score'] = np.clip(classifier_long * ae_familiarity, 0.0, 1.0)
    scored['short_score'] = np.clip(classifier_short * ae_familiarity, 0.0, 1.0)
    scored['long_exit_score'] = np.clip(classifier_long * ae_familiarity, 0.0, 1.0)
    scored['short_exit_score'] = np.clip(classifier_short * ae_familiarity, 0.0, 1.0)
    pred_frames.append(scored)

single_model_scores = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
single_model_scores = single_model_scores.loc[pd.to_datetime(single_model_scores['date']).ge(OOS_START)].copy()
single_model_scores['model_count'] = 1
single_model_scores['net_score'] = single_model_scores['long_score'] - single_model_scores['short_score']

mean_scores = (
    single_model_scores.groupby(['symbol', 'date'], as_index=False)
    .agg(
        long_score=('long_score', 'mean'),
        short_score=('short_score', 'mean'),
        long_exit_score=('long_exit_score', 'mean'),
        short_exit_score=('short_exit_score', 'mean'),
        classifier_long_score=('classifier_long_score', 'mean'),
        classifier_short_score=('classifier_short_score', 'mean'),
        ae_familiarity=('ae_familiarity', 'mean'),
        ae_recon_error=('ae_recon_error', 'mean'),
        ae_latent_distance=('ae_latent_distance', 'mean'),
        model_count=('strategy_source', 'nunique'),
    )
)
mean_scores['source'] = 'ensemble'
mean_scores['family'] = 'mean'
mean_scores['strategy_source'] = 'ensemble_mean'
mean_scores['net_score'] = mean_scores['long_score'] - mean_scores['short_score']

score_cols = ['strategy_source', 'source', 'family', 'symbol', 'date', 'long_score', 'short_score', 'long_exit_score', 'short_exit_score', 'classifier_long_score', 'classifier_short_score', 'ae_familiarity', 'ae_recon_error', 'ae_latent_distance', 'net_score', 'model_count']
strategy_scores = pd.concat(
    [
        mean_scores[score_cols],
        single_model_scores[score_cols],
    ],
    ignore_index=True,
)

print({
    'ensemble_prediction_rows': len(mean_scores),
    'single_model_prediction_rows': len(single_model_scores),
    'strategy_sources': strategy_scores['strategy_source'].nunique(),
    'symbols': strategy_scores['symbol'].nunique(),
    'dates': strategy_scores['date'].nunique(),
})
display(mean_scores[['long_score', 'short_score', 'long_exit_score', 'short_exit_score', 'ae_familiarity', 'ae_latent_distance', 'net_score', 'model_count']].describe())
display(
    strategy_scores.groupby(['strategy_source', 'source', 'family'], as_index=False)
    .agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), dates=('date', 'nunique'), ae_familiarity_mean=('ae_familiarity', 'mean'), ae_latent_distance_mean=('ae_latent_distance', 'mean'))
    .sort_values(['strategy_source'])
)


{'ensemble_prediction_rows': 21151, 'single_model_prediction_rows': 317265, 'strategy_sources': 16, 'symbols': 13, 'dates': 1627}


,long_score,short_score,long_exit_score,short_exit_score,ae_familiarity,ae_latent_distance,net_score,model_count
count,"21,151.0000","21,151.0000","21,151.0000","21,151.0000","21,151.0000","21,151.0000","21,151.0000","21,151.0000"
mean,0.1945,0.2088,0.1945,0.2088,0.4033,0.2929,-0.0144,15.0000
std,0.0552,0.0450,0.0552,0.0450,0.0515,0.1454,0.0865,0.0000
min,0.0632,0.0515,0.0632,0.0515,0.2938,0.0001,-0.4210,15.0000
25%,0.1671,0.1808,0.1671,0.1808,0.3808,0.1869,-0.0590,15.0000
50%,0.1984,0.2066,0.1984,0.2066,0.4017,0.2560,-0.0070,15.0000
75%,0.2230,0.2325,0.2230,0.2325,0.4246,0.3635,0.0370,15.0000
max,0.6486,0.5906,0.6486,0.5906,0.7894,1.0552,0.5079,15.0000


,strategy_source,source,family,rows,symbols,dates,ae_familiarity_mean,ae_latent_distance_mean
0,ensemble_mean,ensemble,mean,21151,13,1627,0.4033,0.2929
1,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,21151,13,1627,0.0130,0.7990
2,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,21151,13,1627,0.0111,0.8390
3,financetoolkit.ft_growth_income,financetoolkit,ft_growth_income,21151,13,1627,0.0133,0.4749
4,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,21151,13,1627,0.0078,0.0237
5,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,21151,13,1627,0.0148,0.0322
6,financetoolkit.ft_ratios_profitability,financetoolkit,ft_ratios_profitability,21151,13,1627,0.0054,0.1081
7,financetoolkit.ft_ratios_solvency,financetoolkit,ft_ratios_solvency,21151,13,1627,0.0042,0.1238
8,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,21151,13,1627,0.0028,0.6479
9,fmp.fmp_balance_mcap,fmp,fmp_balance_mcap,21151,13,1627,0.7174,0.5322


In [9]:
def load_price_frames(symbols):
    frames = {}
    for symbol in symbols:
        prices = WAREHOUSE.read_prices(symbol, provider=FEATURE_CONFIG.provider, start=START_DATE, end=END_DATE)
        if prices is None or prices.empty:
            continue
        frame = prices.rename(columns=str.lower).copy()
        required = ['open', 'high', 'low', 'close', 'volume']
        if not set(required).issubset(frame.columns):
            continue
        frame = frame[required].apply(pd.to_numeric, errors='coerce')
        frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index, errors='coerce')).normalize()
        frame = frame.dropna(subset=['open', 'high', 'low', 'close']).sort_index()
        if not frame.empty:
            frames[str(symbol).upper()] = frame
    return frames

price_frames = load_price_frames(event_symbols)
wide_close = pd.DataFrame({symbol: frame['close'] for symbol, frame in price_frames.items()}).sort_index().ffill()
next_returns = wide_close.pct_change().shift(-1)
print({'price_symbols': wide_close.shape[1], 'price_dates': wide_close.shape[0], 'oos_dates': int((wide_close.index >= OOS_START).sum())})
display(wide_close.tail())


{'price_symbols': 13, 'price_dates': 14109, 'oos_dates': 1627}


,AAPL,AMZN,AVGO,BRK-A,BRK-B,GOOG,GOOGL,LLY,META,MSFT,MU,NVDA,TSLA
date,,,,,,,,,,,,,
2026-06-17,295.9500,237.5000,392.9000,"737,300.0000",491.2800,362.1000,363.7900,"1,112.0000",567.5800,378.9100,"1,043.1900",204.6500,396.3800
2026-06-18,298.0100,244.3900,411.3500,"733,609.7700",489.4600,367.4600,368.0300,"1,098.5700",577.2200,379.4000,"1,133.9900",210.6900,400.4900
2026-06-22,297.0100,232.7900,392.1300,"734,399.9900",488.6900,348.7800,349.6800,"1,102.0800",563.8500,367.3400,"1,211.3800",208.6500,405.0500
2026-06-23,294.3000,234.1100,380.1500,"737,800.0500",492.8100,346.0800,346.1300,"1,107.0800",562.2000,373.9400,"1,051.7700",200.0400,381.6100
2026-06-24,293.0800,234.2700,382.0700,"742,900.0000",494.8100,345.0400,345.2900,"1,117.2600",557.6700,365.4600,"1,048.5100",199.0000,375.5300


## Shared-Book Trading Policy

For every day: exit held positions on the opposite score, then fill open slots from the best current candidates above the entry threshold. Long+short uses one shared `top_k` capacity across both sides.

In [10]:
oos_dates = pd.DatetimeIndex(sorted(set(strategy_scores['date']).intersection(next_returns.index)))
oos_dates = oos_dates[oos_dates >= OOS_START]
trade_symbols = tuple(sorted(set(strategy_scores['symbol']).intersection(next_returns.columns)))
print({'trade_symbols': len(trade_symbols), 'oos_dates': len(oos_dates), 'strategy_sources': strategy_scores['strategy_source'].nunique()})


{'trade_symbols': 13, 'oos_dates': 1627, 'strategy_sources': 16}


In [11]:
weight_artifacts = {}
trade_logs = []
zipline_jobs = []
price_window_start = pd.Timestamp(oos_dates.min()) - pd.Timedelta(days=20)
price_window_end = pd.Timestamp(oos_dates.max())
price_frame_subset = {
    symbol: frame.loc[(pd.DatetimeIndex(frame.index) >= price_window_start) & (pd.DatetimeIndex(frame.index) <= price_window_end)].copy()
    for symbol, frame in price_frames.items()
    if symbol in trade_symbols
}
strategy_source_order = ['ensemble_mean'] + sorted(
    source for source in strategy_scores['strategy_source'].dropna().unique().tolist() if source != 'ensemble_mean'
)

for strategy_source in strategy_source_order:
    score_frame = strategy_scores.loc[strategy_scores['strategy_source'].eq(strategy_source)].copy()
    if score_frame.empty:
        continue
    source_value = score_frame['source'].iloc[0]
    family_value = score_frame['family'].iloc[0]
    print({'strategy_source': strategy_source, 'rows': len(score_frame), 'symbols': score_frame['symbol'].nunique(), 'dates': score_frame['date'].nunique()})
    for variant in ['long_only', 'short_only', 'long_short']:
        for top_k in TOP_K_VALUES:
            weights, trades = build_shared_book_weights(
                score_frame,
                trade_symbols,
                oos_dates,
                top_k=top_k,
                variant=variant,
                entry_threshold=ENTRY_THRESHOLD,
                exit_threshold=EXIT_THRESHOLD,
                long_exit_score_col='long_exit_score',
                short_exit_score_col='short_exit_score',
            )
            weight_artifacts[(strategy_source, variant, top_k)] = weights
            trades = trades.assign(strategy_source=strategy_source, source=source_value, family=family_value, variant=variant, top_k=top_k)
            trade_logs.append(trades)
            zipline_jobs.append(
                ZiplineSharedBookSummaryJob(
                    price_frames=price_frame_subset,
                    target_weights=weights,
                    metadata={
                        'strategy_source': strategy_source,
                        'source': source_value,
                        'family': family_value,
                        'variant': variant,
                        'top_k': top_k,
                        'score_rows': len(score_frame),
                        'score_symbols': score_frame['symbol'].nunique(),
                        'score_dates': score_frame['date'].nunique(),
                        'signal_events': len(trades),
                        'ae_familiarity_mean': float(score_frame['ae_familiarity'].mean()),
                        'ae_familiarity_p25': float(score_frame['ae_familiarity'].quantile(0.25)),
                        'ae_familiarity_p75': float(score_frame['ae_familiarity'].quantile(0.75)),
                        'commission_per_share': ZIPLINE_COMMISSION_PER_SHARE,
                        'slippage_bps': ZIPLINE_SLIPPAGE_BPS,
                        'avg_gross_exposure': float(weights.abs().sum(axis=1).mean()),
                        'avg_net_exposure': float(weights.sum(axis=1).mean()),
                        'fully_invested_days': float(weights.abs().sum(axis=1).ge(0.999).mean()),
                        'cash_days': float(weights.abs().sum(axis=1).eq(0).mean()),
                    },
                    capital_base=CAPITAL_BASE,
                    commission_per_share=ZIPLINE_COMMISSION_PER_SHARE,
                    slippage_bps=ZIPLINE_SLIPPAGE_BPS,
                )
            )

trade_generation_audit = {
    'strategy_sources': len(strategy_source_order),
    'variants': 3,
    'top_k_values': len(TOP_K_VALUES),
    'zipline_jobs': len(zipline_jobs),
    'score_rows_total': int(strategy_scores.groupby('strategy_source').size().sum()),
    'signal_events_total': int(sum(len(frame) for frame in trade_logs)),
    'max_trade_cap': None,
}
expected_jobs = len(strategy_source_order) * 3 * len(TOP_K_VALUES)
assert len(zipline_jobs) == expected_jobs, {'actual_jobs': len(zipline_jobs), 'expected_jobs': expected_jobs}
print({**trade_generation_audit, 'zipline_max_workers': ZIPLINE_MAX_WORKERS})
backtest_summary = run_zipline_shared_book_summary_jobs(zipline_jobs, max_workers=ZIPLINE_MAX_WORKERS)
trade_log = pd.concat(trade_logs, ignore_index=True) if trade_logs else pd.DataFrame()
backtest_summary = backtest_summary.sort_values(['strategy_source', 'variant', 'top_k']).reset_index(drop=True)
metric_cols = ['framework', 'strategy_source', 'source', 'family', 'variant', 'top_k', 'trades', 'signal_events', 'ae_familiarity_mean', 'final_equity', 'total_return', 'annualized_return', 'annualized_vol', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure', 'fully_invested_days', 'cash_days']
display(backtest_summary[metric_cols])
display(trade_log.groupby(['strategy_source', 'variant', 'top_k', 'action'], as_index=False).size().head(80))


{'strategy_source': 'ensemble_mean', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_balance', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_cash', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_income', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_ratios_efficiency', 'rows': 21151, 'symbols': 13, 'dates': 1627}


...

{'strategy_source': 'fmp.fmp_daily_mcap_yield', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'fmp.fmp_income_mcap', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_sources': 16, 'variants': 3, 'top_k_values': 4, 'zipline_jobs': 192, 'score_rows_total': 338416, 'signal_events_total': 41763, 'max_trade_cap': None, 'zipline_max_workers': 4}


,framework,strategy_source,source,family,variant,top_k,trades,signal_events,ae_familiarity_mean,final_equity,total_return,annualized_return,annualized_vol,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,fully_invested_days,cash_days
0,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,5,4556,4,0.4033,"2,067,846.2400",1.0678,0.1191,0.1671,0.7575,-0.2424,0.7999,0.7999,0.0000,0.0000
1,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,10,4277,4,0.4033,"1,462,929.7900",0.4629,0.0607,0.0835,0.7478,-0.1245,0.3999,0.3999,0.0000,0.0000
2,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,20,3718,4,0.4033,"1,214,564.2700",0.2146,0.0306,0.0418,0.7425,-0.0630,0.2000,0.2000,0.0000,0.0000
3,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,40,2901,4,0.4033,"1,103,103.8100",0.1031,0.0153,0.0209,0.7389,-0.0317,0.1000,0.1000,0.0000,0.0000
4,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_short,5,6175,5,0.4033,"1,107,778.2900",0.1078,0.0160,0.1359,0.1847,-0.2409,0.9999,0.5999,0.9994,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,long_short,40,12508,499,0.9176,"792,314.4100",-0.2077,-0.0354,0.0490,-0.7118,-0.2289,0.3249,-0.1076,0.0000,0.0000
188,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,5,5817,83,0.9176,"81,391.7900",-0.9186,-0.3219,0.2819,-1.2370,-0.9267,0.9793,-0.9793,0.9508,0.0000
189,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,10,9553,211,0.9176,"142,838.7300",-0.8572,-0.2602,0.2063,-1.3581,-0.8687,0.8411,-0.8411,0.3485,0.0000
190,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,20,9265,249,0.9176,"382,242.0900",-0.6178,-0.1384,0.1060,-1.3528,-0.6341,0.4325,-0.4325,0.0000,0.0000


,strategy_source,variant,top_k,action,size
0,ensemble_mean,long_only,5,enter_long,4
1,ensemble_mean,long_only,10,enter_long,4
2,ensemble_mean,long_only,20,enter_long,4
3,ensemble_mean,long_only,40,enter_long,4
4,ensemble_mean,long_short,5,enter_long,4
...,...,...,...,...,...
75,financetoolkit.ft_ratios_liquidity,long_short,40,enter_short,3
76,financetoolkit.ft_ratios_liquidity,short_only,5,enter_short,3
77,financetoolkit.ft_ratios_liquidity,short_only,10,enter_short,3
78,financetoolkit.ft_ratios_liquidity,short_only,20,enter_short,3


## Model vs Trading Comparison

In [12]:
model_oos_summary = model_results.loc[model_results['status'].eq('ok'), ['source', 'family', 'oos_rows', 'oos_accuracy', 'oos_balanced_accuracy', 'oos_macro_f1', 'ae_latent_index_rows', 'ae_train_latent_distance_mean', 'ae_train_latent_distance_p95', 'ae_train_error_mean', 'ae_train_error_p95']].copy()
model_oos_summary['strategy_source'] = model_oos_summary.apply(lambda row: strategy_source_name(row['source'], row['family']), axis=1)
display(model_oos_summary.sort_values('oos_macro_f1', ascending=False))

strategy_comparison = (
    backtest_summary.loc[:, ['framework', 'strategy_source', 'source', 'family', 'variant', 'top_k', 'total_return', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure', 'cash_days', 'trades', 'signal_events', 'ae_familiarity_mean']]
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(strategy_comparison)

best_by_strategy_source = (
    strategy_comparison.sort_values(['strategy_source', 'sharpe', 'total_return'], ascending=[True, False, False])
    .groupby('strategy_source', as_index=False)
    .head(1)
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(best_by_strategy_source)

model_vs_trading = best_by_strategy_source.merge(
    model_oos_summary[['strategy_source', 'oos_accuracy', 'oos_balanced_accuracy', 'oos_macro_f1', 'ae_latent_index_rows', 'ae_train_latent_distance_mean', 'ae_train_latent_distance_p95', 'ae_train_error_mean', 'ae_train_error_p95']],
    on='strategy_source',
    how='left',
)
display(model_vs_trading.sort_values(['sharpe', 'total_return'], ascending=False))

baseline_rows = pd.DataFrame([
    {'baseline': 'classifier_only_best_individual', 'strategy_source': 'fmp.fmp_daily_mcap_yield', 'variant': 'long_only', 'top_k': 5, 'total_return': 8.1790, 'sharpe': 1.5984, 'max_drawdown': -0.3022},
    {'baseline': 'classifier_only_best_ensemble', 'strategy_source': 'ensemble_mean', 'variant': 'long_short', 'top_k': 10, 'total_return': 3.0471, 'sharpe': 1.2224, 'max_drawdown': -0.3102},
])
display(baseline_rows)


,source,family,oos_rows,oos_accuracy,oos_balanced_accuracy,oos_macro_f1,ae_latent_index_rows,ae_train_latent_distance_mean,ae_train_latent_distance_p95,ae_train_error_mean,ae_train_error_p95,strategy_source
0,financetoolkit,ft_ratios_valuation,3759,0.4302,0.3953,0.3912,7126,0.0001,0.0000,55.8927,510.2716,financetoolkit.ft_ratios_valuation
1,financetoolkit,ft_ratios_solvency,3759,0.2934,0.3448,0.2676,7126,0.0000,0.0000,35.0185,161.4000,financetoolkit.ft_ratios_solvency
2,fmp,fmp_cash_mcap,3759,0.2969,0.3642,0.2555,6671,0.0363,0.1549,30.7018,110.9803,fmp.fmp_cash_mcap
3,financetoolkit,ft_ratios_efficiency,3759,0.2854,0.3410,0.2515,7126,0.0000,0.0000,4.7516,13.7440,financetoolkit.ft_ratios_efficiency
4,financetoolkit,ft_ratios_profitability,3759,0.2825,0.3402,0.2496,7126,0.0000,0.0000,3.4742,13.1972,financetoolkit.ft_ratios_profitability
5,fmp,fmp_daily_ev_yield,3759,0.2876,0.3533,0.2486,7126,0.0077,0.0287,6.3368,13.7378,fmp.fmp_daily_ev_yield
6,fmp,fmp_daily_mcap_yield,3759,0.2961,0.3635,0.2480,7067,0.0255,0.1016,3.0618,7.8206,fmp.fmp_daily_mcap_yield
7,fmp,fmp_balance_mcap,3759,0.2985,0.3607,0.2423,7067,0.0422,0.1650,17.0206,65.6598,fmp.fmp_balance_mcap
8,fmp,fmp_income_mcap,3759,0.2966,0.3588,0.2413,7126,0.0362,0.1547,13.5925,65.4240,fmp.fmp_income_mcap
9,fmp,fmp_daily_ev_multiple,3759,0.2857,0.3524,0.2413,7063,0.0054,0.0209,22.6749,103.2699,fmp.fmp_daily_ev_multiple


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,cash_days,trades,signal_events,ae_familiarity_mean
0,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,5,4.3476,1.3392,-0.3127,0.6000,0.6000,0.0000,3226,3,0.0078
1,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,5.0238,1.2764,-0.3562,0.8940,0.8940,0.0025,6125,151,0.9330
2,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,10,1.1765,1.2431,-0.1614,0.3000,0.3000,0.0000,3135,3,0.0078
3,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,20,0.4863,1.2402,-0.0828,0.1500,0.1500,0.0000,3016,3,0.0078
4,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,40,0.2216,1.2396,-0.0419,0.0750,0.0750,0.0000,2738,3,0.0078
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,40,-0.3763,-1.3543,-0.3902,0.2163,-0.2163,0.0000,7877,249,0.9176
188,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,10,-0.8572,-1.3581,-0.8687,0.8411,-0.8411,0.0000,9553,211,0.9176
189,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,short_only,40,-0.3470,-1.4738,-0.3661,0.1453,-0.1453,0.0000,5672,32,0.8080
190,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,short_only,20,-0.5792,-1.4747,-0.6031,0.2907,-0.2907,0.0000,6835,32,0.8080


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,cash_days,trades,signal_events,ae_familiarity_mean
0,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,5,4.3476,1.3392,-0.3127,0.6000,0.6000,0.0000,3226,3,0.0078
1,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,5.0238,1.2764,-0.3562,0.8940,0.8940,0.0025,6125,151,0.9330
2,zipline_shared_book_native,fmp.fmp_balance_mcap,fmp,fmp_balance_mcap,long_only,5,3.8703,1.2364,-0.3449,0.9339,0.9339,0.0000,5739,20,0.7174
3,zipline_shared_book_native,fmp.fmp_daily_ev_yield,fmp,fmp_daily_ev_yield,long_only,20,1.2660,1.2283,-0.1893,0.3751,0.3751,0.0000,9832,1483,0.9803
4,zipline_shared_book_native,fmp.fmp_daily_ev_multiple,fmp,fmp_daily_ev_multiple,long_only,10,3.1049,1.1920,-0.3498,0.6793,0.6793,0.0000,9955,1189,0.9044
5,zipline_shared_book_native,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,long_only,10,2.4769,1.0268,-0.3546,0.7000,0.7000,0.0000,10415,7,0.0148
6,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,long_only,5,2.7576,0.9517,-0.4208,0.7572,0.7572,0.0000,5689,163,0.9176
7,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,long_only,10,1.7284,0.9290,-0.2754,0.7118,0.7118,0.0000,9174,29,0.8080
8,zipline_shared_book_native,fmp.fmp_daily_mcap_multiple,fmp,fmp_daily_mcap_multiple,long_only,5,2.1487,0.9118,-0.3435,0.9127,0.9127,0.0000,6206,51,0.7165
9,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_only,5,1.0452,0.8504,-0.2035,0.8000,0.8000,0.0000,4599,4,0.0130


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,cash_days,trades,signal_events,ae_familiarity_mean,oos_accuracy,oos_balanced_accuracy,oos_macro_f1,ae_latent_index_rows,ae_train_latent_distance_mean,ae_train_latent_distance_p95,ae_train_error_mean,ae_train_error_p95
0,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_only,5,4.3476,1.3392,-0.3127,0.6000,0.6000,0.0000,3226,3,0.0078,0.2854,0.3410,0.2515,"7,126.0000",0.0000,0.0000,4.7516,13.7440
1,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,5.0238,1.2764,-0.3562,0.8940,0.8940,0.0025,6125,151,0.9330,0.2961,0.3635,0.2480,"7,067.0000",0.0255,0.1016,3.0618,7.8206
2,zipline_shared_book_native,fmp.fmp_balance_mcap,fmp,fmp_balance_mcap,long_only,5,3.8703,1.2364,-0.3449,0.9339,0.9339,0.0000,5739,20,0.7174,0.2985,0.3607,0.2423,"7,067.0000",0.0422,0.1650,17.0206,65.6598
3,zipline_shared_book_native,fmp.fmp_daily_ev_yield,fmp,fmp_daily_ev_yield,long_only,20,1.2660,1.2283,-0.1893,0.3751,0.3751,0.0000,9832,1483,0.9803,0.2876,0.3533,0.2486,"7,126.0000",0.0077,0.0287,6.3368,13.7378
4,zipline_shared_book_native,fmp.fmp_daily_ev_multiple,fmp,fmp_daily_ev_multiple,long_only,10,3.1049,1.1920,-0.3498,0.6793,0.6793,0.0000,9955,1189,0.9044,0.2857,0.3524,0.2413,"7,063.0000",0.0054,0.0209,22.6749,103.2699
5,zipline_shared_book_native,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,long_only,10,2.4769,1.0268,-0.3546,0.7000,0.7000,0.0000,10415,7,0.0148,0.2748,0.3356,0.2374,"7,126.0000",0.0000,0.0000,22.3406,125.6213
6,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,long_only,5,2.7576,0.9517,-0.4208,0.7572,0.7572,0.0000,5689,163,0.9176,0.2966,0.3588,0.2413,"7,126.0000",0.0362,0.1547,13.5925,65.4240
7,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,long_only,10,1.7284,0.9290,-0.2754,0.7118,0.7118,0.0000,9174,29,0.8080,0.2969,0.3642,0.2555,"6,671.0000",0.0363,0.1549,30.7018,110.9803
8,zipline_shared_book_native,fmp.fmp_daily_mcap_multiple,fmp,fmp_daily_mcap_multiple,long_only,5,2.1487,0.9118,-0.3435,0.9127,0.9127,0.0000,6206,51,0.7165,0.2948,0.3626,0.2395,"7,067.0000",0.0146,0.0624,21.6068,155.9479
9,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_only,5,1.0452,0.8504,-0.2035,0.8000,0.8000,0.0000,4599,4,0.0130,0.2700,0.3314,0.2325,"7,067.0000",0.0005,0.0000,31.8893,125.3346


,baseline,strategy_source,variant,top_k,total_return,sharpe,max_drawdown
0,classifier_only_best_individual,fmp.fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022
1,classifier_only_best_ensemble,ensemble_mean,long_short,10,3.0471,1.2224,-0.3102


In [13]:
baseline_best_individual = {'strategy_source': 'fmp.fmp_daily_mcap_yield', 'variant': 'long_only', 'top_k': 5, 'total_return': 8.1790, 'sharpe': 1.5984, 'max_drawdown': -0.3022}
baseline_best_ensemble = {'strategy_source': 'ensemble_mean', 'variant': 'long_short', 'top_k': 10, 'total_return': 3.0471, 'sharpe': 1.2224, 'max_drawdown': -0.3102}

analysis_lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} FMP 1T+ symbols; {len(event_symbols)} had event coverage.',
    f'- Training window: all available rows through {TRAIN_END.date()}. Out-of-sample model and trading window starts {OOS_START.date()}.',
    f'- Trained classifier + autoencoder feature-family models: {len(models)}.',
    f'- AE device: {AE_DEVICE}; epochs={AE_EPOCHS}; familiarity mode=latent nearest-neighbor reciprocal distance; metric={AE_NN_METRIC}; train-distance percentile={AE_FAMILIARITY_QUANTILE}.',
    f'- Each feature family has its own latent vector index. Total indexed training latent rows: {int(model_results.loc[model_results["status"].eq("ok"), "ae_latent_index_rows"].fillna(0).sum()):,}.',
    f'- Strategy sources traded: {strategy_scores["strategy_source"].nunique()} total, including the ensemble mean and {max(strategy_scores["strategy_source"].nunique() - 1, 0)} individual feature-family models.',
    f'- Entry and exit scores are both classifier probabilities multiplied by AE latent familiarity.',
    f'- Trade generation audit: {trade_generation_audit["zipline_jobs"]} Zipline jobs, {trade_generation_audit["score_rows_total"]:,} score rows across strategy sources, {trade_generation_audit["signal_events_total"]:,} generated signal events, max_trade_cap={trade_generation_audit["max_trade_cap"]}.',
    f'- Strategy variants: long_only, short_only, long_short with top_k={TOP_K_VALUES}; trade size is fixed at 1/top_k and unused slots stay cash.',
    f'- Execution: native Zipline multi-asset shared-book engine using order_target_percent, ${ZIPLINE_COMMISSION_PER_SHARE:.3f}/share commission, and {ZIPLINE_SLIPPAGE_BPS:.1f} bps slippage.',
]
if not model_oos_summary.empty:
    best_model = model_oos_summary.sort_values('oos_macro_f1', ascending=False).iloc[0]
    analysis_lines.append(f'- Best OOS classifier family inside this AE experiment: {best_model["strategy_source"]} with macro_f1={best_model["oos_macro_f1"]:.4f}.')
if not backtest_summary.empty:
    best_trade = backtest_summary.sort_values(['sharpe', 'total_return'], ascending=False).iloc[0]
    ensemble_best = backtest_summary.loc[backtest_summary['strategy_source'].eq('ensemble_mean')].sort_values(['sharpe', 'total_return'], ascending=False).head(1)
    single_best = backtest_summary.loc[~backtest_summary['strategy_source'].eq('ensemble_mean')].sort_values(['sharpe', 'total_return'], ascending=False).head(1)
    analysis_lines.extend([
        '',
        'Best classifier + AE latent-index native Zipline row by Sharpe:',
        f'- {best_trade["strategy_source"]} / {best_trade["variant"]} top_k={int(best_trade["top_k"])}: total_return={best_trade["total_return"]:.2%}, sharpe={best_trade["sharpe"]:.2f}, max_drawdown={best_trade["max_drawdown"]:.2%}, avg_gross={best_trade["avg_gross_exposure"]:.2f}, avg_net={best_trade["avg_net_exposure"]:.2f}, trades={int(best_trade["trades"])}.',
    ])
    if not ensemble_best.empty:
        row = ensemble_best.iloc[0]
        ensemble_sharpe_delta = float(row['sharpe']) - baseline_best_ensemble['sharpe']
        analysis_lines.append(f'- Best classifier + AE ensemble row: {row["variant"]} top_k={int(row["top_k"])} with total_return={row["total_return"]:.2%}, sharpe={row["sharpe"]:.2f}, max_drawdown={row["max_drawdown"]:.2%}; Sharpe delta vs classifier-only ensemble baseline={ensemble_sharpe_delta:+.2f}.')
    if not single_best.empty:
        row = single_best.iloc[0]
        single_sharpe_delta = float(row['sharpe']) - baseline_best_individual['sharpe']
        analysis_lines.append(f'- Best classifier + AE individual row: {row["strategy_source"]} / {row["variant"]} top_k={int(row["top_k"])} with total_return={row["total_return"]:.2%}, sharpe={row["sharpe"]:.2f}, max_drawdown={row["max_drawdown"]:.2%}; Sharpe delta vs classifier-only best individual baseline={single_sharpe_delta:+.2f}.')
    analysis_lines.extend([
        '',
        'Baseline from classifier-only notebook:',
        f'- Best individual classifier-only row: {baseline_best_individual["strategy_source"]} / {baseline_best_individual["variant"]} top_k={baseline_best_individual["top_k"]}: total_return={baseline_best_individual["total_return"]:.2%}, sharpe={baseline_best_individual["sharpe"]:.2f}, max_drawdown={baseline_best_individual["max_drawdown"]:.2%}.',
        f'- Best ensemble classifier-only row: {baseline_best_ensemble["strategy_source"]} / {baseline_best_ensemble["variant"]} top_k={baseline_best_ensemble["top_k"]}: total_return={baseline_best_ensemble["total_return"]:.2%}, sharpe={baseline_best_ensemble["sharpe"]:.2f}, max_drawdown={baseline_best_ensemble["max_drawdown"]:.2%}.',
        '',
        'Interpretation:',
        '- This notebook isolates the latent-index AE question: does closeness to learned pre-2020 latent states improve the classifier trading policy?',
        '- AE familiarity now affects both entries and exits. That makes this test stricter than entry-only gating because unfamiliar exit states can also suppress exit pressure.',
        '- If Sharpe improves while gross exposure or trades fall, latent familiarity is acting as a useful setup-quality filter.',
        '- If Sharpe falls materially, latent familiarity is probably filtering out profitable unfamiliar setups or compressing scores below the 0.5 thresholds.',
        '- Compare model_vs_trading to identify which feature families benefit from latent familiarity and whether latent train-distance diagnostics line up with trading performance.',
    ])
from IPython.display import Markdown, display
display(Markdown('\n'.join(analysis_lines)))


## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained classifier + autoencoder feature-family models: 15.
- AE device: cuda; epochs=20; familiarity mode=latent nearest-neighbor reciprocal distance; metric=euclidean; train-distance percentile=99.9.
- Each feature family has its own latent vector index. Total indexed training latent rows: 105,681.
- Strategy sources traded: 16 total, including the ensemble mean and 15 individual feature-family models.
- Entry and exit scores are both classifier probabilities multiplied by AE latent familiarity.
- Trade generation audit: 192 Zipline jobs, 338,416 score rows across strategy sources, 41,763 generated signal events, max_trade_cap=None.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Execution: native Zipline multi-asset shared-book engine using order_target_percent, $0.005/share commission, and 5.0 bps slippage.
- Best OOS classifier family inside this AE experiment: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best classifier + AE latent-index native Zipline row by Sharpe:
- financetoolkit.ft_ratios_efficiency / long_only top_k=5: total_return=434.76%, sharpe=1.34, max_drawdown=-31.27%, avg_gross=0.60, avg_net=0.60, trades=3226.
- Best classifier + AE ensemble row: long_only top_k=5 with total_return=106.78%, sharpe=0.76, max_drawdown=-24.24%; Sharpe delta vs classifier-only ensemble baseline=-0.46.
- Best classifier + AE individual row: financetoolkit.ft_ratios_efficiency / long_only top_k=5 with total_return=434.76%, sharpe=1.34, max_drawdown=-31.27%; Sharpe delta vs classifier-only best individual baseline=-0.26.

Baseline from classifier-only notebook:
- Best individual classifier-only row: fmp.fmp_daily_mcap_yield / long_only top_k=5: total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.
- Best ensemble classifier-only row: ensemble_mean / long_short top_k=10: total_return=304.71%, sharpe=1.22, max_drawdown=-31.02%.

Interpretation:
- This notebook isolates the latent-index AE question: does closeness to learned pre-2020 latent states improve the classifier trading policy?
- AE familiarity now affects both entries and exits. That makes this test stricter than entry-only gating because unfamiliar exit states can also suppress exit pressure.
- If Sharpe improves while gross exposure or trades fall, latent familiarity is acting as a useful setup-quality filter.
- If Sharpe falls materially, latent familiarity is probably filtering out profitable unfamiliar setups or compressing scores below the 0.5 thresholds.
- Compare model_vs_trading to identify which feature families benefit from latent familiarity and whether latent train-distance diagnostics line up with trading performance.

## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained classifier + autoencoder feature-family models: 15.
- AE device: cuda; epochs=20; familiarity mode=latent nearest-neighbor reciprocal distance; metric=euclidean; train-distance percentile=99.9.
- Each feature family has its own latent vector index. Total indexed training latent rows: 105,681.
- Strategy sources traded: 16 total, including the ensemble mean and 15 individual feature-family models.
- Entry and exit scores are both classifier probabilities multiplied by AE latent familiarity.
- Trade generation audit: 192 Zipline jobs, 338,416 score rows across strategy sources, 41,763 generated signal events, max_trade_cap=None.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Execution: native Zipline multi-asset shared-book engine using order_target_percent, $0.005/share commission, and 5.0 bps slippage.
- Best OOS classifier family inside this AE experiment: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best classifier + AE latent-index native Zipline row by Sharpe:
- financetoolkit.ft_ratios_efficiency / long_only top_k=5: total_return=434.76%, sharpe=1.34, max_drawdown=-31.27%, avg_gross=0.60, avg_net=0.60, trades=3226.
- Best classifier + AE ensemble row: long_only top_k=5 with total_return=106.78%, sharpe=0.76, max_drawdown=-24.24%; Sharpe delta vs classifier-only ensemble baseline=-0.46.
- Best classifier + AE individual row: financetoolkit.ft_ratios_efficiency / long_only top_k=5 with total_return=434.76%, sharpe=1.34, max_drawdown=-31.27%; Sharpe delta vs classifier-only best individual baseline=-0.26.

Baseline from classifier-only notebook:
- Best individual classifier-only row: fmp.fmp_daily_mcap_yield / long_only top_k=5: total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.
- Best ensemble classifier-only row: ensemble_mean / long_short top_k=10: total_return=304.71%, sharpe=1.22, max_drawdown=-31.02%.

Interpretation:
- This notebook isolates the latent-index AE question: does closeness to learned pre-2020 latent states improve the classifier trading policy?
- AE familiarity now affects both entries and exits. That makes this test stricter than entry-only gating because unfamiliar exit states can also suppress exit pressure.
- If Sharpe improves while gross exposure or trades fall, latent familiarity is acting as a useful setup-quality filter.
- If Sharpe falls materially, latent familiarity is probably filtering out profitable unfamiliar setups or compressing scores below the 0.5 thresholds.
- Compare model_vs_trading to identify which feature families benefit from latent familiarity and whether latent train-distance diagnostics line up with trading performance.